# 09 — Deployment Optimization

Optimize trained models for on-device inference: classical feature imputation (reduce extraction cost), spectrogram resolution reduction for CNNs, TensorFlow Lite quantization (float16/int8), and latency/size benchmarking. Produce a comparison table to inform deployment choice (classical vs neural).


In [4]:
# Imports and setup
from pathlib import Path
import numpy as np, pandas as pd, time, yaml, joblib
import tensorflow as tf
from tensorflow import keras
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
CFG_PATH = PROJECT_ROOT / 'config.yaml'
CLASSICAL_DIR = PROJECT_ROOT / 'data' / 'processed' / 'classical_features'
NEURAL_DIR = PROJECT_ROOT / 'data' / 'processed' / 'neural_features'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'metrics'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
for d in [RESULTS_DIR, FIG_DIR]: d.mkdir(parents=True, exist_ok=True)
print('TF:', tf.__version__)


TF: 2.14.0


In [5]:
# Load classical features and tuned models
df = pd.read_csv(CLASSICAL_DIR / 'classical_features.csv')
label_col = 'label'; path_col = 'path' if 'path' in df.columns else 'segment_path'
y_str = df[label_col].values
X_full = df.drop(columns=[label_col, path_col] if path_col in df.columns else [label_col]).values
le = joblib.load(MODELS_DIR / 'classical' / 'label_encoder.pkl')
y = le.transform(y_str)
# Train/test split consistent with Notebook 05 (reload indices if saved); fallback to simple split
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y, test_size=0.2, stratify=y, random_state=42)
scaler = joblib.load(MODELS_DIR / 'classical' / 'scaler.pkl')
X_tr_sc = scaler.transform(X_tr); X_te_sc = scaler.transform(X_te)
svm = joblib.load(MODELS_DIR / 'classical' / 'svm_tuned.pkl') if (MODELS_DIR / 'classical' / 'svm_tuned.pkl').exists() else None
rf  = joblib.load(MODELS_DIR / 'classical' / 'rf_tuned.pkl') if (MODELS_DIR / 'classical' / 'rf_tuned.pkl').exists() else None
knn = joblib.load(MODELS_DIR / 'classical' / 'knn_tuned.pkl') if (MODELS_DIR / 'classical' / 'knn_tuned.pkl').exists() else None
xgb = joblib.load(MODELS_DIR / 'classical' / 'xgb_tuned.pkl') if (MODELS_DIR / 'classical' / 'xgb_tuned.pkl').exists() else None
print('Loaded classical models available:', [m for m in ['svm','rf','knn','xgb'] if eval(m) is not None])


Loaded classical models available: ['svm', 'knn']


## Classical: Feature Imputation (62 → 194)

In [ ]:
# Split features into 'available' (fast) and 'expensive' per plan (example: first 62 vs remaining)
n_fast = 62
X_fast_tr, X_fast_te = X_tr_sc[:, :n_fast], X_te_sc[:, :n_fast]
X_exp_tr,  X_exp_te  = X_tr_sc[:, n_fast:], X_te_sc[:, n_fast:]
imputer = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1))
t0 = time.time(); imputer.fit(X_fast_tr, X_exp_tr); t_fit = time.time() - t0
joblib.dump(imputer, MODELS_DIR / 'classical' / 'rf_imputer.pkl')
X_exp_pred = imputer.predict(X_fast_te)
X_full_imputed = np.hstack([X_fast_te, X_exp_pred])
print('Imputer fit time (s):', round(t_fit, 3))
# Evaluate tuned models on imputed features
def eval_model(name, model, X, y):
    if model is None: return None
    t0 = time.time(); y_pred = model.predict(X); t_inf = (time.time()-t0)/len(y)
    acc = accuracy_score(y, y_pred)
    return {'model': name, 'accuracy': acc, 'inf_time_ms_per_sample': t_inf*1000}
results_imp = []
for name, mdl in [('SVM', svm), ('RF', rf), ('KNN', knn), ('XGB', xgb)]:
    r_full = eval_model(name+' full', mdl, X_te_sc, y_te)
    r_imp  = eval_model(name+' imputed', mdl, X_full_imputed, y_te)
    if r_full: results_imp.append(r_full)
    if r_imp:  results_imp.append(r_imp)
df_imp = pd.DataFrame(results_imp); df_imp.to_csv(RESULTS_DIR / 'imputation_benchmark.csv', index=False)
df_imp


## Neural: Spectrogram Resolution Reduction

In [ ]:
# Recompute reduced spectrograms (50,20) as a quick deployable alternative
X1_path = NEURAL_DIR / 'spectrograms_1ch.npy'
X3_path = NEURAL_DIR / 'spectrograms_3ch.npy'
if X1_path.exists():
    X1 = np.load(X1_path)  # (N, 99, 40, 1)
    # Downsample time/mel by simple slicing (prototype; for production recompute with librosa)
    X1_small = X1[:, ::2, ::2, :]  # ~ (50,20)
    np.save(NEURAL_DIR / 'spectrograms_1ch_small.npy', X1_small)
if X3_path.exists():
    X3 = np.load(X3_path)
    X3_small = X3[:, ::2, ::2, :]
    np.save(NEURAL_DIR / 'spectrograms_3ch_small.npy', X3_small)
print('Saved reduced spectrograms if sources existed.')


## TFLite Quantization (Custom CNN)

In [ ]:
# Load best CNN and convert to TFLite with float16 + int8 (with representative dataset)
CNN_DIR = PROJECT_ROOT / 'models' / 'neural'
best_model_path = CNN_DIR / 'custom_cnn_best.keras'
if best_model_path.exists():
    model = tf.keras.models.load_model(best_model_path)
    # Float16 quantization
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.target_spec.supported_types = [tf.float16]
    tflite_f16 = conv.convert()
    (PROJECT_ROOT / 'models' / 'tflite').mkdir(parents=True, exist_ok=True)
    with open(PROJECT_ROOT / 'models' / 'tflite' / 'custom_cnn_f16.tflite', 'wb') as f: f.write(tflite_f16)
    print('Saved float16 TFLite model.')
    
    # Int8 quantization with representative dataset
    # Build a small generator from a subset of train spectrograms
    rep_data = np.load(NEURAL_DIR / 'spectrograms_1ch.npy') if (NEURAL_DIR / 'spectrograms_1ch.npy').exists() else np.load(NEURAL_DIR / 'spectrograms_3ch.npy')
    def rep_gen():
        for i in range(0, min(200, len(rep_data)), 5):
            yield [rep_data[i:i+1].astype(np.float32)]
    conv2 = tf.lite.TFLiteConverter.from_keras_model(model)
    conv2.optimizations = [tf.lite.Optimize.DEFAULT]
    conv2.representative_dataset = rep_gen
    conv2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv2.inference_input_type = tf.uint8 if model.input.dtype==tf.uint8 else tf.int8
    conv2.inference_output_type = tf.uint8 if model.output.dtype==tf.uint8 else tf.int8
    try:
        tflite_int8 = conv2.convert()
        with open(PROJECT_ROOT / 'models' / 'tflite' / 'custom_cnn_int8.tflite', 'wb') as f: f.write(tflite_int8)
        print('Saved int8 TFLite model.')
    except Exception as e:
        print('Int8 conversion skipped:', e)
else:
    print('Best CNN model not found—run Notebook 07 first.')


## TFLite Inference Benchmark

In [ ]:
# Benchmark TFLite models on desktop (simulated)
import numpy as np, time
def tflite_infer_time(tflite_path, input_shape):
    if not Path(tflite_path).exists(): return None
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    dummy = np.random.randn(*input_shape).astype(np.float32)
    t0 = time.time()
    for _ in range(50):
        interpreter.set_tensor(input_details[0]['index'], dummy)
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details[0]['index'])
    return (time.time()-t0)/50 * 1000  # ms per inference

f16_path = PROJECT_ROOT / 'models' / 'tflite' / 'custom_cnn_f16.tflite'
int8_path = PROJECT_ROOT / 'models' / 'tflite' / 'custom_cnn_int8.tflite'
# Infer input shape from saved arrays
if (NEURAL_DIR / 'spectrograms_1ch.npy').exists():
    sample = np.load(NEURAL_DIR / 'spectrograms_1ch.npy')[0:1]
else:
    sample = np.load(NEURAL_DIR / 'spectrograms_3ch.npy')[0:1]
ms_f16 = tflite_infer_time(f16_path, sample.shape)
ms_int8 = tflite_infer_time(int8_path, sample.shape)
print('TFLite inference ms/sample — float16:', ms_f16, 'int8:', ms_int8)


## Comparison Table and Save

In [ ]:
# Compose final comparison for deployment decision
rows = []
def add_row(name, acc=None, f1=None, params=None, size_kb=None, inf_ms=None):
    rows.append({'model': name, 'accuracy': acc, 'f1_macro': f1, 'params': params, 'size_kb': size_kb, 'inf_time_ms': inf_ms})

# Classical (using test set)
for nm, mdl in [('SVM tuned', svm), ('RF tuned', rf), ('KNN tuned', knn), ('XGB tuned', xgb)]:
    if mdl is not None:
        t0=time.time(); yhat=mdl.predict(X_te_sc); inf=(time.time()-t0)/len(y_te)*1000
        acc=accuracy_score(y_te, yhat)
        add_row(nm, acc=acc, f1=None, params=None, size_kb=None, inf_ms=inf)

# Neural (Keras model sizes)
cnn_best = PROJECT_ROOT / 'models' / 'neural' / 'custom_cnn_best.keras'
if cnn_best.exists():
    mdl = tf.keras.models.load_model(cnn_best)
    params = mdl.count_params()
    add_row('Custom CNN (float32)', params=params)
    f16 = PROJECT_ROOT / 'models' / 'tflite' / 'custom_cnn_f16.tflite'
    if f16.exists(): add_row('Custom CNN (TFLite f16)', size_kb=f16.stat().st_size/1024, inf_ms=ms_f16)
    i8 = PROJECT_ROOT / 'models' / 'tflite' / 'custom_cnn_int8.tflite'
    if i8.exists(): add_row('Custom CNN (TFLite int8)', size_kb=i8.stat().st_size/1024, inf_ms=ms_int8)

df_cmp = pd.DataFrame(rows)
df_cmp.to_csv(RESULTS_DIR / 'deployment_comparison.csv', index=False)
df_cmp
